In [1]:
import pandas as pd 
multi_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT/RE_TACRED_full_binary_result.csv')

In [2]:
multi_df.columns[3:]

Index(['token', 'subject', 'subj_start', 'subj_end', 'subj_type', 'object',
       'obj_start', 'obj_end', 'obj_type', 'relation',
       ...
       'occupation_title', 'title_held_by_person', 'title_holder',
       'referent_of', 'title_holder_of', 'held_position',
       'marital_partner_of_blogger', 'profession_of_person',
       'title_or_position_holder', 'title_assignment'],
      dtype='object', length=817)

In [3]:
multi_df.head(2)

,split,sample_id,text,token,subject,subj_start,subj_end,subj_type,object,obj_start,...,occupation_title,title_held_by_person,title_holder,referent_of,title_holder_of,held_position,marital_partner_of_blogger,profession_of_person,title_or_position_holder,title_assignment
0,train,61b3a65fb9acc5c72e88,"* `` Updates , Dcr -LRB- DCR -RRB- '' < Dcr.Up...","['*', '``', 'Updates', ',', 'Dcr', '-LRB-', 'D...",DCR,6,6,ORGANIZATION,Dcr,4,...,0,0,0,1,0,0,0,0,0,0
1,train,61b3a65fb916e324f16c,-LRB- While I 'm sure that Fidelity also reser...,"['-LRB-', 'While', 'I', ""'m"", 'sure', 'that', ...",Fidelity,14,14,ORGANIZATION,Fidelity,6,...,0,0,0,1,0,0,0,0,0,0


In [8]:
type(multi_df['multi_labels'][1])

list

In [9]:
import ast 

multi_df = multi_df.rename(columns={'gpt4_phrase': 'predicted_relation'})


labels_series = multi_df['multi_labels'].explode() 
labels_count = labels_series.value_counts().reset_index() 
labels_count.columns = ['predicted_relation', 'num_samples'] 

relation_to_ids = ( 
    multi_df
        .explode('multi_labels')                        
        .groupby('multi_labels')['sample_id']                  
        .apply(list)                                    
        .to_dict()                                      
)


labels_count['id_list'] = labels_count['predicted_relation'].map(relation_to_ids) 

relations = labels_count['predicted_relation'].tolist() 


for rel in relations: 
    labels_count[rel] = 0
    

id_sets = [set(ids) for ids in labels_count['id_list']]


for i, rel_i in enumerate(relations): 
    ids_i = id_sets[i] 
    for j, rel_j in enumerate(relations): 
        ids_j = id_sets[j] 
        overlap = len(ids_i & ids_j)   
        labels_count.at[i, rel_j] = overlap
labels_count.head(2)

/tmp/ipykernel_2396416/3192595657.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labels_count[rel] = 0
/tmp/ipykernel_2396416/3192595657.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labels_count[rel] = 0
/tmp/ipykernel_2396416/3192595657.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

,predicted_relation,num_samples,id_list,attribute_of,ownership_or_affiliation,subject_involved_in_object,involvement_in,affiliated_with,affiliation_origin,affiliation_with,...,diplomatic_appeal_to_judicial_authorities,convicted_for_supporting_rebels,political_candidacy_in_country,legal_restriction_on_political_candidacy,airline_operates_in_country,incorrectlycreditedwithfirstafricanamericanchairmanship,company-has-experienced-growth-in-dividend,expelled_from_organization,launch_site_of_space_mission_operator,has_matching_tattoos_with
0,attribute_of,1170,"[61b3a65fb9acc5c72e88, e7798a9e3660ae176967, 6...",1170,757,736,693,646,667,584,...,0,1,1,1,2,1,0,0,0,0
1,ownership_or_affiliation,1061,"[61b3a65fb9acc5c72e88, e7798a9e3660ae176967, 6...",757,1061,708,747,782,688,728,...,2,2,2,1,1,1,1,1,1,1


In [10]:
labels_count.to_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT/RE_TACRED_full_intersection_matrix.csv', index=False)